# 02_baseline_grid.ipynb

## XGBoost + SHAP Baseline for Grid (Stateless) Experiment

This notebook:
1. Trains XGBoost on the IEEE 118-bus grid data
2. Predicts voltage at all 118 buses
3. Computes SHAP values for feature importance
4. Compares results with LLMIP (to be done in notebook 03)

## 1. Setup

In [14]:
import pandas as pd
import numpy as np
import json
from pathlib import Path
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import xgboost as xgb
import shap
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.multioutput import MultiOutputRegressor

print("Libraries loaded successfully!")

Libraries loaded successfully!


## 2. Load Data

In [16]:
PROJECT_DIR = Path("/home/l1zle/LLMIP")
DATA_DIR = PROJECT_DIR / "data" / "prepared"
RESULTS_DIR = PROJECT_DIR / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Load grid data
train_features = pd.read_csv(DATA_DIR / "llmip_train_features.csv", index_col=0)
test_features = pd.read_csv(DATA_DIR / "llmip_test_features.csv", index_col=0)
train_targets = pd.read_csv(DATA_DIR / "llmip_train_targets.csv", index_col=0)
test_targets = pd.read_csv(DATA_DIR / "llmip_test_targets.csv", index_col=0)

# Filter to vm_pu columns only
vm_cols = [c for c in train_targets.columns if c.startswith('vm_')]
train_targets = train_targets[vm_cols]
test_targets = test_targets[vm_cols]

print("=== Data Loaded ===")
print(f"Train features: {train_features.shape}")
print(f"Test features:  {test_features.shape}")
print(f"Train targets:  {train_targets.shape} (118 bus voltages)")
print(f"Test targets:   {test_targets.shape}")

=== Data Loaded ===
Train features: (160, 78)
Test features:  (40, 78)
Train targets:  (160, 118) (118 bus voltages)
Test targets:   (40, 118)


## 3. Define Bus Categories

In [17]:
# Generator buses (control voltage, operate at higher pu)
GENERATOR_BUSES = [1, 4, 6, 8, 10, 12, 15, 18, 24, 25, 26, 31, 32, 34, 36, 40, 42, 46, 49, 54, 55, 56, 59, 61, 65, 66, 69, 70, 72, 73, 74, 76, 77, 79, 80, 82, 85, 87, 89, 91, 92, 99, 100, 103, 104, 105, 107, 110, 111, 112, 113, 116]

# Load buses (dependent on load, operate near 1.0 pu)
LOAD_BUSES = [b for b in range(1, 119) if b not in GENERATOR_BUSES]

print(f"Generator buses: {len(GENERATOR_BUSES)}")
print(f"Load buses:     {len(LOAD_BUSES)}")
print(f"Total:          {len(GENERATOR_BUSES) + len(LOAD_BUSES)}")

Generator buses: 52
Load buses:     66
Total:          118


## 4. Train XGBoost Model (Multi-Output)

In [18]:
print("\n=== Training Multi-Output XGBoost ===")
print(f"Train: {len(train_features)} samples, {len(train_features.columns)} features")
print(f"Predicting: {len(train_targets.columns)} bus voltages")

# Multi-output regressor - one XGBoost model per bus
base_model = xgb.XGBRegressor(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    random_state=42,
    tree_method='hist',
)

model = MultiOutputRegressor(base_model)
model.fit(train_features, train_targets)

print("\nModel trained successfully!")


=== Training Multi-Output XGBoost ===
Train: 160 samples, 78 features
Predicting: 118 bus voltages

Model trained successfully!


## 5. Make Predictions

In [19]:
print("\n=== Making Predictions on Test Set ===")

predictions = model.predict(test_features)
predictions_df = pd.DataFrame(predictions, columns=train_targets.columns, index=test_targets.index)

print(f"Predictions shape: {predictions_df.shape}")
print(f"Test targets shape: {test_targets.shape}")


=== Making Predictions on Test Set ===
Predictions shape: (40, 118)
Test targets shape: (40, 118)


## 6. Compute Error Metrics

In [20]:
# Compute per-bus and overall metrics
errors = np.abs(predictions - test_targets.values)
mae_per_bus = pd.Series(errors.mean(axis=0), index=train_targets.columns, dtype=np.float64)
rmse_per_bus = pd.Series(np.sqrt((errors ** 2).mean(axis=0)), index=train_targets.columns, dtype=np.float64)

overall_mae = mae_per_bus.mean()
overall_rmse = np.sqrt((errors ** 2).mean())
    
    # Calculate MAPE (Mean Absolute Percentage Error)
    # MAPE = mean(|actual - pred| / |actual|) * 100
    with np.errstate(divide='ignore', invalid='ignore'):
        pct_errors = np.abs(errors) / np.abs(test_targets.values)
        pct_errors = pct_errors[np.isfinite(pct_errors)]  # Remove inf/nan
        mape = np.mean(pct_errors) * 100

# Error by bus type
gen_mae = mae_per_bus[[f"vm_{b}" for b in GENERATOR_BUSES if f"vm_{b}" in mae_per_bus]].mean()
load_mae = mae_per_bus[[f"vm_{b}" for b in LOAD_BUSES if f"vm_{b}" in mae_per_bus]].mean()

print("\n=== XGBoost Performance Results ===")
print(f"\n  Overall MAE:  {overall_mae:.4f} pu")
print(f"  Overall RMSE: {overall_rmse:.4f} pu")
    print(f"  Overall MAPE: {mape:.2f}%")
print(f"\n  Generator bus MAE:  {gen_mae:.4f} pu")
print(f"  Load bus MAE:       {load_mae:.4f} pu")
print(f"\n  Best bus MAE:  {mae_per_bus.min():.4f} pu  ({mae_per_bus.idxmin()})")
print(f"  Worst bus MAE: {mae_per_bus.max():.4f} pu  ({mae_per_bus.idxmax()})")


=== XGBoost Performance Results ===

  Overall MAE:  0.0015 pu
  Overall RMSE: 0.0039 pu

  Generator bus MAE:  0.0014 pu
  Load bus MAE:       0.0017 pu

  Best bus MAE:  0.0000 pu  (vm_25)
  Worst bus MAE: 0.0073 pu  (vm_18)


In [21]:
# Per-bus error statistics
print("\n=== Per-Bus MAE Statistics ===")
print(f"  Median: {mae_per_bus.median():.4f}")
print(f"  Std:    {mae_per_bus.std():.4f}")
print(f"  P25:    {mae_per_bus.quantile(0.25):.4f}")
print(f"  P75:    {mae_per_bus.quantile(0.75):.4f}")


=== Per-Bus MAE Statistics ===
  Median: 0.0011
  Std:    0.0013
  P25:    0.0007
  P75:    0.0022


## 7. Compute SHAP Values

In [22]:
print("\n=== Computing SHAP Values ===")
print("This may take a moment...")

# Use subset of test samples for SHAP
n_shap_samples = min(50, len(test_features))
shap_X = test_features.head(n_shap_samples)

# Compute SHAP for each bus model
shap_values_all = []
for i, estimator in enumerate(model.estimators_):
    shap_vals = shap.TreeExplainer(estimator).shap_values(shap_X)
    shap_values_all.append(shap_vals)
    if (i + 1) % 20 == 0:
        print(f"  Processed {i + 1}/{len(model.estimators_)} buses...")

shap_values_arr = np.array(shap_values_all)
shap_abs = np.abs(shap_values_arr)

print(f"\nSHAP computed for {len(shap_values_all)} buses × {n_shap_samples} samples")


=== Computing SHAP Values ===
This may take a moment...
  Processed 20/118 buses...
  Processed 40/118 buses...
  Processed 60/118 buses...
  Processed 80/118 buses...
  Processed 100/118 buses...

SHAP computed for 118 buses × 40 samples


## 8. Feature Importance (Aggregated)

In [23]:
# Aggregate SHAP across all buses
shap_aggregated = shap_abs.mean(axis=1).mean(axis=0)
top_features = pd.Series(shap_aggregated, index=train_features.columns).sort_values(ascending=False)

print("\n=== Top 10 Features (Aggregated SHAP) ===")
for feat, val in top_features.head(10).items():
    print(f"  {feat:30s} {val:.6f}")


=== Top 10 Features (Aggregated SHAP) ===
  gen_wind_q_r3_mvar             0.000789
  gen_hydro_q_r1_mvar            0.000757
  gen_thermal_q_r3_mvar          0.000749
  gen_solar_q_r1_mvar            0.000524
  load_p_r1_mw                   0.000485
  gen_thermal_q_r1_mvar          0.000471
  slack_vm_pu                    0.000409
  load_p_r2_mw                   0.000381
  gen_total_q_r1_mvar            0.000378
  gen_solar_p_r3_mw              0.000339


In [24]:
# Feature importance by bus type
print("\n=== Top Features by Bus Type ===")

# SHAP per bus
shap_per_bus = {}
for bus_idx, bus_col in enumerate(train_targets.columns):
    bus_shap = shap_abs[bus_idx].mean(axis=0)
    shap_per_bus[bus_col] = pd.Series(bus_shap, index=train_features.columns).sort_values(ascending=False).to_dict()

# Generator bus SHAP
gen_buses = [f"vm_{b}" for b in GENERATOR_BUSES if f"vm_{b}" in train_targets.columns]
gen_shap_df = pd.DataFrame(shap_per_bus)
gen_shap_agg = gen_shap_df[gen_buses].mean(axis=1).sort_values(ascending=False)

print("\n  Generator buses - Top 5:")
for feat, val in gen_shap_agg.head(5).items():
    print(f"    {feat:30s} {val:.6f}")

# Load bus SHAP
load_buses = [f"vm_{b}" for b in LOAD_BUSES if f"vm_{b}" in train_targets.columns]
load_shap_df = pd.DataFrame(shap_per_bus)
load_shap_agg = load_shap_df[load_buses].mean(axis=1).sort_values(ascending=False)

print("\n  Load buses - Top 5:")
for feat, val in load_shap_agg.head(5).items():
    print(f"    {feat:30s} {val:.6f}")


=== Top Features by Bus Type ===

  Generator buses - Top 5:
    gen_solar_p_r3_mw              0.000553
    slack_vm_pu                    0.000470
    gen_hydro_q_r1_mvar            0.000464
    load_p_r1_mw                   0.000347
    gen_thermal_q_r3_mvar          0.000323

  Load buses - Top 5:
    gen_wind_q_r3_mvar             0.001220
    gen_thermal_q_r3_mvar          0.001085
    gen_hydro_q_r1_mvar            0.000989
    gen_solar_q_r1_mvar            0.000733
    load_p_r1_mw                   0.000595


## 9. Visualizations

In [29]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1. MAE per bus
ax1 = axes[0, 0]
mae_sorted = mae_per_bus.sort_values()
colors = ['#f39c12' if int(str(m).replace('vm_', '')) in GENERATOR_BUSES else '#3498db'
          for m in mae_sorted.index]
ax1.barh(range(len(mae_sorted)), mae_sorted.values, color=colors, height=1.0)
ax1.set_yticks([])
ax1.set_xlabel('MAE (pu)')
ax1.set_title('XGBoost MAE per Bus\n(orange=generator, blue=load)')
ax1.axvline(overall_mae, color='red', linestyle='--', label=f'Mean={overall_mae:.4f}')
ax1.legend()

# 2. Top features SHAP
ax2 = axes[0, 1]
top_n = 10
shap_top = top_features.head(top_n)
ax2.barh(range(top_n), shap_top.values[::-1], color='#9b59b6')
ax2.set_yticks(range(top_n))
ax2.set_yticklabels(shap_top.index[::-1], fontsize=9)
ax2.set_xlabel('Mean |SHAP|')
ax2.set_title('Top 10 Features by SHAP (aggregated)')

# 3. Error distribution histogram
ax3 = axes[1, 0]
errors_flat = errors.flatten()
ax3.hist(errors_flat, bins=50, edgecolor='black', alpha=0.7, color='steelblue')
ax3.axvline(overall_mae, color='red', linestyle='--', label=f'MAE={overall_mae:.4f}')
ax3.set_xlabel('Absolute Error (pu)')
ax3.set_ylabel('Frequency')
ax3.set_title('Distribution of Prediction Errors')
ax3.legend()

# 4. Actual vs Predicted (sample bus)
ax4 = axes[1, 1]
sample_bus = 'vm_1'
ax4.scatter(test_targets[sample_bus], predictions_df[sample_bus], alpha=0.7, s=50)
ax4.plot([test_targets[sample_bus].min(), test_targets[sample_bus].max()],
         [test_targets[sample_bus].min(), test_targets[sample_bus].max()],
         'r--', label='Perfect')
ax4.set_xlabel(f'Actual {sample_bus} (pu)')
ax4.set_ylabel(f'Predicted {sample_bus} (pu)')
ax4.set_title(f'Actual vs Predicted ({sample_bus})')
ax4.legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'xgboost_grid_baseline.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFigure saved to: {RESULTS_DIR / 'xgboost_grid_baseline.png'}")


Figure saved to: /home/l1zle/LLMIP/results/xgboost_grid_baseline.png


/tmp/ipykernel_7498/1542432553.py:49: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 10. Save Results

In [26]:
# Save results to JSON
results = {
    "domain": "grid",
    "model": "MultiOutputRegressor(XGBRegressor)",
    "n_buses": int(len(train_targets.columns)),
    "n_train": int(len(train_features)),
    "n_test": int(len(test_features)),
    "n_features": int(len(train_features.columns)),
    "generator_buses": GENERATOR_BUSES,
    "load_buses": LOAD_BUSES,
    "metrics": {
        "mae_overall": float(overall_mae),
        "rmse_overall": float(overall_rmse),
        "mae_generator_buses": float(gen_mae),
        "mae_load_buses": float(load_mae),
        "mae_per_bus": mae_per_bus.to_dict(),
        "rmse_per_bus": rmse_per_bus.to_dict(),
        "mae_best_bus": mae_per_bus.idxmin(),
        "mae_worst_bus": mae_per_bus.idxmax(),
    },
    "shap": {
        "n_samples_used": n_shap_samples,
        "top_features_aggregated": top_features.head(20).to_dict(),
        "top_features_generator_buses": gen_shap_agg.head(10).to_dict(),
        "top_features_load_buses": load_shap_agg.head(10).to_dict(),
    },
}

with open(RESULTS_DIR / 'xgboost_grid_results.json', 'w') as f:
    json.dump(results, f, indent=2)

print(f"Results saved to: {RESULTS_DIR / 'xgboost_grid_results.json'}")

Results saved to: /home/l1zle/LLMIP/results/xgboost_grid_results.json


## 11. Summary

In [27]:
print("\n" + "="*60)
print("XGBOOST BASELINE COMPLETE")
print("="*60)
print("\n=== Performance Summary ===")
print(f"  Overall MAE:  {overall_mae:.4f} pu")
print(f"  Overall RMSE: {overall_rmse:.4f} pu")
print(f"  Generator bus MAE:  {gen_mae:.4f} pu")
print(f"  Load bus MAE:       {load_mae:.4f} pu")

print("\n=== Top 5 Most Important Features ===")
for i, (feat, val) in enumerate(top_features.head(5).items()):
    print(f"  {i+1}. {feat:30s} ({val:.6f})")

print("\n=== Output Files ===")
print(f"  Results JSON: {RESULTS_DIR / 'xgboost_grid_results.json'}")
print(f"  Figure:       {RESULTS_DIR / 'xgboost_grid_baseline.png'}")

print("\n=== Next Steps ===")
print("  1. Run 03_llmip_grid.ipynb to run the LLMIP pipeline")
print("  2. Compare Replicability Scores between XGBoost and LLMIP")


XGBOOST BASELINE COMPLETE

=== Performance Summary ===
  Overall MAE:  0.0015 pu
  Overall RMSE: 0.0039 pu
  Generator bus MAE:  0.0014 pu
  Load bus MAE:       0.0017 pu

=== Top 5 Most Important Features ===
  1. gen_wind_q_r3_mvar             (0.000789)
  2. gen_hydro_q_r1_mvar            (0.000757)
  3. gen_thermal_q_r3_mvar          (0.000749)
  4. gen_solar_q_r1_mvar            (0.000524)
  5. load_p_r1_mw                   (0.000485)

=== Output Files ===
  Results JSON: /home/l1zle/LLMIP/results/xgboost_grid_results.json
  Figure:       /home/l1zle/LLMIP/results/xgboost_grid_baseline.png

=== Next Steps ===
  1. Run 03_llmip_grid.ipynb to run the LLMIP pipeline
  2. Compare Replicability Scores between XGBoost and LLMIP
